# 🥗 Gorilla Fitness Recipe Book - Trainee Diet Plan Generator
### Lab 3: Retrieval-Augmented Generation (RAG) & Structured Schema Extraction

This notebook demonstrates how to load a PDF recipe book (`gorilla-fitness-recipe-book.pdf`), extract and index its nutritional recipes using **LangChain** and **FAISS**, calculate customized energy & macronutrient targets (TDEE/BMR) for a fitness trainee, and use an LLM (**Groq**) with **Structured Output Schemas** to generate an end-to-end personalized diet plan.

---

### 📋 Pipeline Architecture:
1. **Environment & Dependency Setup**: Install and load LangChain, Groq, PyPDF, FAISS, and sentence transformers.
2. **Document Ingestion**: Read `gorilla-fitness-recipe-book.pdf` using `PyPDFLoader`.
3. **Text Chunking & Vector Indexing**: Split recipe pages into chunks and build a FAISS vector store.
4. **Trainee Profiling & Macro Calculation**: Calculate Mifflin-St Jeor BMR, TDEE, and macro splits (fat loss, muscle gain, maintenance).
5. **Structured Output Schemas**: Define `ResponseSchema` and `StructuredOutputParser` for strict JSON outputs.
6. **RAG & LLM Diet Plan Synthesis**: Retrieve relevant recipes and prompt Groq LLM to assemble the daily plan.
7. **Report Formatting**: Pretty-print the meal plan, macro totals, grocery shopping list, and coach recommendations.
8. **Reusable Trainee Pipeline**: A one-call function to generate diet plans for any trainee profile.

## 1. Install & Import Required Libraries

In [1]:
# Install required packages (if not already installed)
!pip install langchain langchain-groq langchain-huggingface langchain-community pypdf faiss-cpu sentence-transformers langchain-classic pydantic

In [2]:
import os
import re
import sys
import json
from typing import List, Dict, Any, Optional

# LangChain Document Processing
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector Stores & Embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# LLM & Prompt Templates
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# Structured Output Parsers (Lab 3 Schema Approach)
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser

print("All libraries imported successfully.")

All libraries imported successfully.


<string>:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


## 2. API Key Configuration
Set your Groq API key to power the LLM inference.

In [3]:
# Configure Groq API Key
# You can get a free key from: https://console.groq.com/
if "GROQ_API_KEY" not in os.environ or not os.environ["GROQ_API_KEY"]:
    os.environ["GROQ_API_KEY"] = ""

print("Groq API Key status:", "CONFIGURED" if os.environ.get("GROQ_API_KEY") else "MISSING")


Groq API Key status: CONFIGURED


## 3. Load & Read the Recipe Book PDF
Load the Gorilla Fitness Recipe Book (`gorilla-fitness-recipe-book.pdf`).

In [4]:
# Path to the recipe book PDF
pdf_path = r"E:\LLMs\gorilla-fitness-recipe-book.pdf"

print(f"Loading recipe book from: {pdf_path}")
loader = PyPDFLoader(pdf_path)
docs = loader.load()

print(f"\u2705 Successfully loaded {len(docs)} pages.")

# Inspect first page preview
if docs:
    print("\n--- First Page Preview ---")
    print(docs[0].page_content[:400] + "...")

Loading recipe book from: E:\LLMs\gorilla-fitness-recipe-book.pdf
✅ Successfully loaded 46 pages.

--- First Page Preview ---
...


## 4. Text Chunking & Vector Store (RAG Setup)
We chunk the recipe book content so individual recipes, ingredients, and instructions can be retrieved semantically.

In [5]:
# Chunk the recipe documents
# Chunk size keeps recipe ingredients and instructions cohesive
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)

splits = text_splitter.split_documents(docs)
print(f"\u2705 Generated {len(splits)} chunks from the recipe book.")

✅ Generated 83 chunks from the recipe book.


In [6]:
# Initialize Embeddings and FAISS Vector Store
# Includes a graceful fallback in case local Windows AppControl blocks torch.dll
try:
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(splits, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    print("\u2705 FAISS Vector Store and Embeddings initialized successfully.")
except Exception as e:
    print(f"\u26a0\ufe0f Vectorstore warning: {e}")
    print("Initializing lightweight keyword recipe retriever fallback...")
    class KeywordRecipeRetriever:
        def __init__(self, documents):
            self.documents = documents
        def invoke(self, query: str):
            keywords = [w.lower() for w in query.split() if len(w) > 2]
            scored = []
            for d in self.documents:
                content = d.page_content.lower()
                score = sum(1 for kw in keywords if kw in content)
                scored.append((score, d))
            scored.sort(key=lambda x: x[0], reverse=True)
            return [d for s, d in scored[:5]]
    retriever = KeywordRecipeRetriever(splits)
    print("\u2705 Fallback retriever ready.")

✅ FAISS Vector Store and Embeddings initialized successfully.


Loading weights: 100%|##########| 103/103 [00:00<00:00, 2530.88it/s]


In [7]:
# Test recipe retrieval with a sample query
test_query = "high protein chicken breakfast or lunch"
retrieved_samples = retriever.invoke(test_query)

print(f"Found {len(retrieved_samples)} matching chunks for: '{test_query}'\n")
for i, item in enumerate(retrieved_samples[:2], 1):
    page_num = item.metadata.get("page", "N/A")
    print(f"[Result {i} - Page {page_num}]")
    print(item.page_content[:250].strip() + "...\n")

Found 5 matching chunks for: 'high protein chicken breakfast or lunch'

[Result 1 - Page 5]
. pineapple chunks for a simple,  carbs, 5g fat 
. refreshing meal. 
Chicken and 100g grilled chicken, 1/2 cup cooked 
- uinoa , 1 cup mixed greens (30g), Quinoa Salad 229?5;9 T mvocatis (4001 1 tap e o8 199) 
Combine chicken, quinoa, and 400 kcal, 3...

[Result 2 - Page 21]
: : 5 
iy . 
Chicken breast (grilled)": 100, 'Lino 
oat pasta (cooked)': 100, 'Pesto": 30, 4 
'Olive oil': 10, 'Chenry tomatoes": 40, / 
'Parmesan': 10 
* . Pesto Pasta 
with Grilled 
Chicken agﬂed 
Girill chicken; toss pasta with 
pesto and olive oi...



## 5. Trainee Profile & Nutritional Target Calculator
A fitness diet plan must be calibrated to the trainee's Basal Metabolic Rate (BMR), Total Daily Energy Expenditure (TDEE), and specific goal (Fat Loss / Cutting, Muscle Gain / Bulking, or Maintenance).

In [8]:
def calculate_trainee_targets(
    weight_kg: float,
    height_cm: float,
    age: int,
    gender: str,
    activity_level: str = "moderate",
    goal: str = "fat_loss"
) -> Dict[str, Any]:
    """
    Calculates BMR via Mifflin-St Jeor, TDEE, and optimal daily macros.
    """
    # 1. BMR Calculation
    if gender.lower() == "male":
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age + 5
    else:
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age - 161

    # 2. Activity Multiplier
    activity_factors = {
        "sedentary": 1.2,        # Desk job, little/no exercise
        "light": 1.375,          # Light exercise 1-3 days/week
        "moderate": 1.55,        # Moderate exercise 3-5 days/week
        "very_active": 1.725,    # Hard training 6-7 days/week
        "extra_active": 1.9      # Intense physical job & training
    }
    tdee = bmr * activity_factors.get(activity_level.lower(), 1.55)

    # 3. Target Calories & Macro Split based on fitness goal
    if goal == "fat_loss":
        target_cals = round(tdee - 500)             # Moderate 500 kcal deficit
        protein_g = round(2.2 * weight_kg)          # 2.2g per kg for muscle retention
        fats_g = round((target_cals * 0.25) / 9)    # 25% healthy fats
        carbs_g = round((target_cals - (protein_g * 4 + fats_g * 9)) / 4)
    elif goal == "muscle_gain":
        target_cals = round(tdee + 350)             # Clean 350 kcal surplus
        protein_g = round(2.0 * weight_kg)
        fats_g = round((target_cals * 0.25) / 9)
        carbs_g = round((target_cals - (protein_g * 4 + fats_g * 9)) / 4)
    else:  # maintenance
        target_cals = round(tdee)
        protein_g = round(1.8 * weight_kg)
        fats_g = round((target_cals * 0.28) / 9)
        carbs_g = round((target_cals - (protein_g * 4 + fats_g * 9)) / 4)

    return {
        "bmr": round(bmr),
        "tdee": round(tdee),
        "target_calories": target_cals,
        "protein_g": protein_g,
        "carbs_g": max(40, carbs_g),
        "fats_g": fats_g
    }

# Define sample trainee profile
trainee = {
    "name": "Ahmed Tarek",
    "age": 26,
    "gender": "male",
    "weight_kg": 84.0,
    "height_cm": 182.0,
    "activity_level": "moderate",     # sedentary, light, moderate, very_active
    "goal": "fat_loss",               # fat_loss, muscle_gain, maintenance
    "dietary_restrictions": "No seafood, prefers chicken and eggs, high protein",
    "meals_per_day": 4                # Breakfast, Lunch, Snack, Dinner
}

# Calculate targets
targets = calculate_trainee_targets(
    weight_kg=trainee["weight_kg"],
    height_cm=trainee["height_cm"],
    age=trainee["age"],
    gender=trainee["gender"],
    activity_level=trainee["activity_level"],
    goal=trainee["goal"]
)
trainee.update(targets)

print("\u2705 Trainee Profile & Calibrated Targets:")
print(json.dumps(trainee, indent=2))

✅ Trainee Profile & Calibrated Targets:
{
  "name": "Ahmed Tarek",
  "age": 26,
  "gender": "male",
  "weight_kg": 84.0,
  "height_cm": 182.0,
  "activity_level": "moderate",
  "goal": "fat_loss",
  "dietary_restrictions": "No seafood, prefers chicken and eggs, high protein",
  "meals_per_day": 4,
  "bmr": 1852,
  "tdee": 2871,
  "target_calories": 2371,
  "protein_g": 185,
  "carbs_g": 259,
  "fats_g": 66
}


# Initialize Groq Chat Model
# High performance open-source models: openai/gpt-oss-120b, openai/gpt-oss-20b
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    api_key=os.environ.get("GROQ_API_KEY")
)

# Formulate tailored search queries to pull a rich variety of meals from the recipe book
search_queries = [
    f"breakfast recipes high protein {trainee['dietary_restrictions']}",
    f"lunch recipes healthy calories {trainee['goal']}",
    f"dinner recipes high protein low carb",
    f"healthy snack post workout"
]

retrieved_context_docs = []
seen_content = set()
for q in search_queries:
    matches = retriever.invoke(q)
    for m in matches:
        content_sample = m.page_content[:100]
        if content_sample not in seen_content:
            seen_content.add(content_sample)
            retrieved_context_docs.append(m)

# Format recipes context for prompt
recipes_context = "\n\n---\n\n".join([
    f"[Recipe Book Page {doc.metadata.get('page', 'N/A')}]:\n{doc.page_content}"
    for doc in retrieved_context_docs[:8]
])

print(f"✅ Retrieved {len(retrieved_context_docs)} recipe excerpts for context.")


In [9]:
# Define Schemas for the Diet Plan Output
summary_schema = ResponseSchema(
    name="trainee_summary",
    description="Summary of the trainee: name, goal, target calories, and macro breakdown (protein_g, carbs_g, fats_g)."
)

meals_schema = ResponseSchema(
    name="meals",
    description=(
        "List of meals planned for the day strictly extracted from the recipe book. Each meal item must contain: "
        "'meal_type' (e.g. Breakfast, Lunch, Snack, Dinner), "
        "'recipe_name' (exact or matching recipe from the book), "
        "'page_reference' (page number in book if found, else null), "
        "'key_ingredients' (list of ingredients with portion sizes), "
        "'calories' (estimated kcal), "
        "'protein_g' (grams of protein), "
        "'carbs_g' (grams of carbs), "
        "'fats_g' (grams of fats), "
        "'preparation_notes' (brief cooking / prep summary from the recipe book)."
    )
)

totals_schema = ResponseSchema(
    name="daily_totals",
    description="Object containing total calories, total protein_g, total carbs_g, total fats_g, and adherence comparison with trainee targets."
)

grocery_schema = ResponseSchema(
    name="grocery_list",
    description="Categorized grocery list of all ingredients needed for the day's meals (e.g. Proteins, Produce, Dairy, Grains, Spices)."
)

coaching_schema = ResponseSchema(
    name="coaching_advice",
    description="List of professional sports nutritionist recommendations (meal timing, workout window, water intake, adherence)."
)

diet_schemas = [
    summary_schema,
    meals_schema,
    totals_schema,
    grocery_schema,
    coaching_schema
]

output_parser = StructuredOutputParser.from_response_schemas(diet_schemas)
format_instructions = output_parser.get_format_instructions()

print("\u2705 Structured Output Parser initialized.")
print("Format instructions sample preview:\n", format_instructions[:300] + "...")

✅ Structured Output Parser initialized.
Format instructions sample preview:
 The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"trainee_summary": string  // Summary of the trainee: name, goal, target calories, and macro breakdown (protein_g, carbs_g, fats_g).
	"meals": string  /...


## 7. RAG Retrieval & Groq LLM Diet Plan Generation
Retrieve the most relevant recipes from the book matching the trainee's calorie target and dietary preferences, then prompt the LLM to synthesize the plan.

In [10]:
# Initialize Groq Chat Model
# High performance open-source models: llama-3.3-70b-versatile, llama-3.1-8b-instant, or openai/gpt-oss-120b
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2
)

# Formulate tailored search queries to pull a rich variety of meals from the recipe book
search_queries = [
    f"breakfast recipes high protein {trainee['dietary_restrictions']}",
    f"lunch recipes healthy calories {trainee['goal']}",
    f"dinner recipes high protein low carb",
    f"healthy snack post workout"
]

retrieved_context_docs = []
seen_content = set()
for q in search_queries:
    matches = retriever.invoke(q)
    for m in matches:
        content_sample = m.page_content[:100]
        if content_sample not in seen_content:
            seen_content.add(content_sample)
            retrieved_context_docs.append(m)

# Format recipes context for prompt
recipes_context = "\n\n---\n\n".join([
    f"[Recipe Book Page {doc.metadata.get('page', 'N/A')}]:\n{doc.page_content}"
    for doc in retrieved_context_docs[:8]
])

print(f"\u2705 Retrieved {len(retrieved_context_docs)} recipe excerpts for context.")

✅ Retrieved 14 recipe excerpts for context.


In [11]:
# Construct prompt and call Groq in JSON mode with openai/gpt-oss-120b
import os
import json
from groq import Groq

# 1. Build prompt explicitly enforcing the structured JSON schema with nested objects
prompt = f'''
You are an expert sports nutritionist and personal trainer.
Create a customized 1-day meal plan for the trainee below strictly using recipes from the Gorilla Fitness Recipe Book context.

TRAINEE INFO:
{json.dumps(trainee, indent=2)}

AVAILABLE RECIPES FROM BOOK:
{recipes_context}

CRITICAL: Return STRICTLY a valid JSON object formatted EXACTLY as follows (all meals must be items in a JSON array, NOT strings):
```json
{{
  "trainee_summary": {{
    "name": "{trainee['name']}",
    "goal": "{trainee['goal']}",
    "target_calories": {trainee['target_calories']},
    "protein_g": {trainee['protein_g']},
    "carbs_g": {trainee['carbs_g']},
    "fats_g": {trainee['fats_g']}
  }},
  "meals": [
    {{
      "meal_type": "Breakfast",
      "recipe_name": "Recipe Name From Book",
      "page_reference": 31,
      "key_ingredients": ["ingredient 1 with portion", "ingredient 2 with portion"],
      "calories": 450,
      "protein_g": 35,
      "carbs_g": 30,
      "fats_g": 15,
      "preparation_notes": "Preparation instructions from the recipe book."
    }}
  ],
  "daily_totals": {{
    "total_calories": {trainee['target_calories']},
    "total_protein_g": {trainee['protein_g']},
    "total_carbs_g": {trainee['carbs_g']},
    "total_fats_g": {trainee['fats_g']},
    "adherence_comparison": "Brief comment on how close the plan matches targets"
  }},
  "grocery_list": {{
    "Proteins": ["item 1", "item 2"],
    "Produce": ["item 1", "item 2"],
    "Grains & Pantry": ["item 1", "item 2"]
  }},
  "coaching_advice": [
    "Practical coach tip 1 regarding water or timing",
    "Practical coach tip 2 regarding pre/post workout"
  ]
}}
```
Do not include any conversational preamble or text outside the JSON object.
'''

# Alias prompt_text to prompt for compatibility
prompt_text = prompt

# 2. Call Groq with JSON mode and low reasoning effort
groq_api_key = os.environ.get("GROQ_API_KEY", "")
client = Groq(api_key=groq_api_key)

completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    response_format={"type": "json_object"},
    temperature=1,
    max_completion_tokens=4096,
    top_p=1,
    reasoning_effort="low",
    stream=True,
    stop=None
)

raw_response_content = ""
for chunk in completion:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="", flush=True)
    raw_response_content += delta

print("\n\n✅ LLM Response streamed and captured successfully.")


{
  "trainee_summary": {
    "name": "Ahmed Tarek",
    "goal": "fat_loss",
    "target_calories": 2371,
    "protein_g": 185,
    "carbs_g": 259,
    "fats_g": 66
  },
  "meals": [
    {
      "meal_type": "Breakfast",
      "recipe_name": "Protein Pancakes",
      "page_reference": 31,
      "key_ingredients": [
        "1 scoop Limitless Alpha whey protein",
        "1/2 cup Lino oats (40g)",
        "1 egg",
        "1/2 tsp baking powder",
        "1/4 cup almond milk"
      ],
      "calories": 460,
      "protein_g": 45,
      "carbs_g": 42,
      "fats_g": 12,
      "preparation_notes": "Blend all ingredients into a smooth batter and cook pancakes on a non‑stick pan. Top with fresh berries and serve warm."
    },
    {
      "meal_type": "Lunch",
      "recipe_name": "Chicken Quinoa Salad",
      "page_reference": 5,
      "key_ingredients": [
        "100 g grilled chicken breast",
        "1/2 cup cooked quinoa",
        "1 cup mixed greens",
        "1/2 avocado",
        "1

## 8. Output Parsing & Professional Report Display
Parse the structured JSON output and format it into a clear, aesthetic diet plan report.

In [12]:
def extract_and_parse_plan(text: str, parser: StructuredOutputParser = None) -> Dict[str, Any]:
    """
    Robustly extract and parse JSON from the LLM response.
    """
    text = (text or "").strip()
    if not text:
        raise ValueError("Model returned an empty response.")

    plan = None

    # 1. Direct JSON parse
    try:
        plan = json.loads(text)
    except Exception:
        pass

    # 2. Extract between first '{' and last '}'
    if plan is None:
        start = text.find('{')
        end = text.rfind('}')
        if start != -1 and end != -1 and end > start:
            try:
                plan = json.loads(text[start:end + 1].strip())
            except Exception:
                pass

    # 3. Extract from markdown code block
    if plan is None:
        match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
        if match:
            try:
                plan = json.loads(match.group(1).strip())
            except Exception:
                pass

    # 4. Fallback to parser
    if plan is None and parser is not None:
        try:
            plan = parser.parse(text)
        except Exception:
            pass

    if plan is None or not isinstance(plan, dict):
        raise ValueError(f"Could not extract a valid JSON plan from response (Length: {len(text)} chars).\nContent: {text[:200]}")

    # Ensure nested objects are parsed if returned as strings
    for k in ["meals", "grocery_list", "daily_totals", "trainee_summary", "coaching_advice"]:
        if k in plan and isinstance(plan[k], str):
            try:
                plan[k] = json.loads(plan[k])
            except Exception:
                if k == "meals":
                    found = []
                    for m_match in re.finditer(r"\{.*?\}", plan[k], re.DOTALL):
                        try:
                            found.append(json.loads(m_match.group(0)))
                        except Exception:
                            pass
                    if found:
                        plan[k] = found

    return plan

def get_plan(llm_model, prompt_input, parser: StructuredOutputParser = None) -> Dict[str, Any]:
    """
    End-to-end plan generation and parsing function.
    Supports PromptTemplate, ChatPromptTemplate, or raw string prompt.
    """
    # If raw_response_content is already populated from the previous cell, use it directly
    global raw_response_content
    if 'raw_response_content' in globals() and raw_response_content:
        return extract_and_parse_plan(raw_response_content, parser)
    
    # Otherwise invoke model
    if hasattr(prompt_input, "format"):
        rendered = prompt_input.format()
    elif hasattr(prompt_input, "to_string"):
        rendered = prompt_input.to_string()
    else:
        rendered = str(prompt_input)

    try:
        from groq import Groq
        key = os.environ.get("GROQ_API_KEY", "")
        cl = Groq(api_key=key)
        comp = cl.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": rendered}],
            response_format={"type": "json_object"},
            temperature=1,
            max_completion_tokens=4096,
            reasoning_effort="low"
        )
        content = comp.choices[0].message.content
    except Exception:
        content = llm_model.invoke(rendered).content

    return extract_and_parse_plan(content, parser)

# Generate and parse plan
parsed_plan = get_plan(llm, prompt, output_parser)
print("✅ Successfully parsed plan. Sections:", list(parsed_plan.keys()))


✅ Successfully parsed plan. Sections: ['trainee_summary', 'meals', 'daily_totals', 'grocery_list', 'coaching_advice']


In [13]:
def print_diet_plan_report(plan: Dict[str, Any]):
    """
    Renders a comprehensive, formatted meal plan report for the trainee.
    """
    border = "=" * 80
    print(border)
    print("🏋️‍♂️  GORILLA FITNESS TRAINEE DIET PLAN REPORT  🏋️‍♂️")
    print(border)

    # 1. Trainee Summary
    summary = plan.get("trainee_summary", {})
    if isinstance(summary, dict):
        print(f"👤 Trainee: {summary.get('name', 'N/A')}")
        print(f"🎯 Goal:    {summary.get('goal', 'N/A')}")
        print(f"🔥 Targets: {summary.get('target_calories', 'N/A')} kcal | "
              f"Protein: {summary.get('protein_g', summary.get('protein', 'N/A'))}g | "
              f"Carbs: {summary.get('carbs_g', summary.get('carbs', 'N/A'))}g | "
              f"Fats: {summary.get('fats_g', summary.get('fats', 'N/A'))}g")
    else:
        print(f"👤 Trainee Summary: {summary}")

    print("-" * 80)
    print("🍽️ DAILY MEAL SCHEDULE (SOURCED FROM RECIPE BOOK):")
    print("-" * 80)

    # 2. Meals Breakdown
    meals = plan.get("meals", [])
    if isinstance(meals, list):
        for i, m in enumerate(meals, 1):
            if isinstance(m, dict):
                m_type = m.get("meal_type", f"Meal {i}")
                r_name = m.get("recipe_name", "Fitness Recipe")
                page = m.get("page_reference")
                page_str = f" (Page {page})" if page else ""
                cals = m.get("calories", "N/A")
                p = m.get("protein_g", "N/A")
                c = m.get("carbs_g", "N/A")
                f = m.get("fats_g", "N/A")
                prep = m.get("preparation_notes", "")

                print(f"\n[{m_type.upper()}] 🍳 {r_name}{page_str}")
                print(f"   📊 Macros:      {cals} kcal | Protein: {p}g | Carbs: {c}g | Fats: {f}g")

                ingredients = m.get("key_ingredients", [])
                if isinstance(ingredients, list):
                    print(f"   🥗 Ingredients: {', '.join(str(ing) for ing in ingredients)}")
                elif ingredients:
                    print(f"   🥗 Ingredients: {ingredients}")

                if prep:
                    print(f"   📝 Prep Notes:  {prep}")
            else:
                print(f"\nMeal {i}: {m}")
    else:
        print(f"{meals}")

    # 3. Totals Comparison
    print("\n" + "-" * 80)
    print("📈 DAILY NUTRITION TOTALS:")
    totals = plan.get("daily_totals", {})
    if isinstance(totals, dict):
        print(f"   Total Energy:  {totals.get('total_calories', totals.get('calories', 'N/A'))} kcal")
        print(f"   Total Protein: {totals.get('total_protein_g', totals.get('protein', 'N/A'))}g")
        print(f"   Total Carbs:   {totals.get('total_carbs_g', totals.get('carbs', 'N/A'))}g")
        print(f"   Total Fats:    {totals.get('total_fats_g', totals.get('fats', 'N/A'))}g")
        if "adherence_comparison" in totals or "difference_from_target" in totals:
            diff = totals.get("adherence_comparison") or totals.get("difference_from_target")
            print(f"   Adherence:     {diff}")
    else:
        print(f"   {totals}")

    # 4. Grocery List
    print("-" * 80)
    print("🛒 SHOPPING GROCERY LIST:")
    groceries = plan.get("grocery_list", {})
    if isinstance(groceries, dict):
        for category, items in groceries.items():
            items_text = ", ".join(items) if isinstance(items, list) else str(items)
            print(f"   • {category.title()}: {items_text}")
    elif isinstance(groceries, list):
        for it in groceries:
            print(f"   • {it}")
    else:
        print(f"   {groceries}")

    # 5. Coaching Advice
    print("-" * 80)
    print("💡 COACHING & NUTRITION TIPS:")
    advice = plan.get("coaching_advice", [])
    if isinstance(advice, list):
        for tip in advice:
            print(f"   ✓ {tip}")
    else:
        print(f"   {advice}")

    print(border)

print_diet_plan_report(parsed_plan)


🏋️‍♂️  GORILLA FITNESS TRAINEE DIET PLAN REPORT  🏋️‍♂️
👤 Trainee: Ahmed Tarek
🎯 Goal:    fat_loss
🔥 Targets: 2371 kcal | Protein: 185g | Carbs: 259g | Fats: 66g
--------------------------------------------------------------------------------
🍽️ DAILY MEAL SCHEDULE (SOURCED FROM RECIPE BOOK):
--------------------------------------------------------------------------------

[BREAKFAST] 🍳 Protein Pancakes (Page 31)
   📊 Macros:      460 kcal | Protein: 45g | Carbs: 42g | Fats: 12g
   🥗 Ingredients: 1 scoop Limitless Alpha whey protein, 1/2 cup Lino oats (40g), 1 egg, 1/2 tsp baking powder, 1/4 cup almond milk
   📝 Prep Notes:  Blend all ingredients into a smooth batter and cook pancakes on a non‑stick pan. Top with fresh berries and serve warm.

[LUNCH] 🍳 Chicken Quinoa Salad (Page 5)
   📊 Macros:      620 kcal | Protein: 38g | Carbs: 45g | Fats: 28g
   🥗 Ingredients: 100 g grilled chicken breast, 1/2 cup cooked quinoa, 1 cup mixed greens, 1/2 avocado, 1 tbsp olive oil
   📝 Prep Notes:  C

## 9. Reusable Diet Plan Function for Any Trainee
This modular function lets you generate a customized diet plan for any trainee with a single function call.

In [14]:
def create_trainee_diet_plan(
    name: str,
    age: int,
    gender: str,
    weight_kg: float,
    height_cm: float,
    goal: str = "fat_loss",             # 'fat_loss', 'muscle_gain', 'maintenance'
    activity_level: str = "moderate",   # 'sedentary', 'light', 'moderate', 'very_active'
    dietary_restrictions: str = "None",
    meals_per_day: int = 4
) -> Dict[str, Any]:
    """
    Generates a complete personalized diet plan for any trainee using
    the Gorilla Fitness Recipe Book context and Groq LLM.
    """
    # 1. Calculate caloric and macro targets
    user_targets = calculate_trainee_targets(
        weight_kg=weight_kg,
        height_cm=height_cm,
        age=age,
        gender=gender,
        activity_level=activity_level,
        goal=goal
    )

    profile = {
        "name": name,
        "age": age,
        "gender": gender,
        "weight_kg": weight_kg,
        "height_cm": height_cm,
        "goal": goal,
        "activity_level": activity_level,
        "dietary_restrictions": dietary_restrictions,
        "meals_per_day": meals_per_day,
        **user_targets
    }

    # 2. Retrieve recipes from vector store
    retrieved = retriever.invoke(f"healthy {goal} recipes {dietary_restrictions} target {user_targets['target_calories']} kcal")
    context = "\n\n".join([f"[Page {d.metadata.get('page', 'N/A')}]: {d.page_content}" for d in retrieved])

    # 3. Build Prompt
    custom_prompt = f"""
    Design a 1-day personalized meal plan for this trainee using recipes from the Gorilla Fitness Recipe Book:
    Trainee Profile: {json.dumps(profile, indent=2)}

    Recipes Context:
    {context}

    Ensure total calories match ~{user_targets['target_calories']} kcal and protein ~{user_targets['protein_g']}g.

    {format_instructions}
    """

    # 4. Invoke LLM and Parse
    result = llm.invoke(custom_prompt)
    return extract_and_parse_plan(result.content, output_parser)

# Example of running for another trainee (e.g. Muscle Gain / Bulking):
print("Reusable generator ready. Example usage:")
print("""
# bulking_plan = create_trainee_diet_plan(
#     name="Youssef Ali",
#     age=24,
#     gender="male",
#     weight_kg=72.0,
#     height_cm=176.0,
#     goal="muscle_gain",
#     dietary_restrictions="High carbs and protein, no dairy",
#     meals_per_day=4
# )
# print_diet_plan_report(bulking_plan)
""")

Reusable generator ready. Example usage:

# bulking_plan = create_trainee_diet_plan(
#     name="Youssef Ali",
#     age=24,
#     gender="male",
#     weight_kg=72.0,
#     height_cm=176.0,
#     goal="muscle_gain",
#     dietary_restrictions="High carbs and protein, no dairy",
#     meals_per_day=4
# )
# print_diet_plan_report(bulking_plan)

